# Fault Tolerance Benchmark Analysis
**Author:** Rajshekar Medipally  
**Project:** spark-streaming-fault-tolerance  
**GitHub:** https://github.com/rmedipallycic/spark-streaming-fault-tolerance

This notebook analyzes 24 benchmark experiments evaluating fault tolerance and exactly-once delivery semantics in Apache Spark Structured Streaming across three checkpoint strategies, four failure scenarios, and three storage backends.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_csv('../experiments/summary.csv')

# Clean up
df['recovery_latency_sec'] = pd.to_numeric(df['recovery_latency_sec'], errors='coerce')
df['throughput_degradation_pct'] = pd.to_numeric(df['throughput_degradation_pct'], errors='coerce')
df['duplicates_detected'] = pd.to_numeric(df['duplicates_detected'], errors='coerce').fillna(0)

# Strategy labels
strategy_labels = {
    'A': 'Strategy A\n(High-Frequency)',
    'B': 'Strategy B\n(Interval-Based)',
    'C': 'Strategy C\n(Async WAL)'
}
df['strategy_label'] = df['strategy'].map(strategy_labels)

print(f'Loaded {len(df)} experiments.')
df.head()

## 2. Throughput Comparison — Baseline (Normal Operation)

In [ ]:
baseline = df[df['scenario'] == 'baseline'].copy()
local_baseline = baseline[baseline['checkpoint_type'] == 'local']

fig, ax = plt.subplots()

bars = ax.bar(
    local_baseline['strategy_label'],
    local_baseline['normal_throughput_rps'],
    color=['#4C72B0', '#DD8452', '#55A868'],
    width=0.5, edgecolor='white', linewidth=1.2
)

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f'{bar.get_height():,.0f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Baseline Throughput by Checkpoint Strategy (Local Storage)', fontsize=14, pad=15)
ax.set_ylabel('Records per Second', fontsize=12)
ax.set_xlabel('Checkpoint Strategy', fontsize=12)
ax.set_ylim(0, local_baseline['normal_throughput_rps'].max() * 1.2)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('../experiments/fig1_baseline_throughput.png', dpi=150, bbox_inches='tight')
plt.show()
print('Finding: Strategy B (Interval-Based) achieves highest throughput under normal operation.')

## 3. Recovery Latency by Failure Scenario

In [ ]:
recovery = df[
    (df['scenario'] != 'baseline') &
    (df['checkpoint_type'] == 'local') &
    (df['recovery_latency_sec'].notna())
].copy()

pivot = recovery.pivot_table(
    index='scenario',
    columns='strategy',
    values='recovery_latency_sec'
)

fig, ax = plt.subplots()
pivot.plot(kind='bar', ax=ax, width=0.7, edgecolor='white', colormap='muted')

ax.set_title('Recovery Latency by Failure Scenario and Checkpoint Strategy', fontsize=14, pad=15)
ax.set_ylabel('Recovery Latency (seconds)', fontsize=12)
ax.set_xlabel('Failure Scenario', fontsize=12)
ax.legend(['Strategy A', 'Strategy B', 'Strategy C'], title='Strategy')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../experiments/fig2_recovery_latency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Finding: Strategy C (Async WAL) consistently achieves lowest recovery latency.')

## 4. Throughput Degradation Under Failure

In [ ]:
degradation = df[
    (df['scenario'] != 'baseline') &
    (df['checkpoint_type'] == 'local') &
    (df['throughput_degradation_pct'].notna())
].copy()

fig, ax = plt.subplots()
for i, (strategy, group) in enumerate(degradation.groupby('strategy')):
    ax.scatter(
        group['scenario'],
        group['throughput_degradation_pct'],
        label=f'Strategy {strategy}',
        s=120, zorder=3
    )

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Throughput Degradation (%) by Failure Scenario', fontsize=14, pad=15)
ax.set_ylabel('Throughput Degradation (%)', fontsize=12)
ax.set_xlabel('Failure Scenario', fontsize=12)
ax.legend(title='Strategy')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('../experiments/fig3_throughput_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Storage Backend Comparison — Network Partition

In [ ]:
network = df[df['scenario'] == 'network_partition'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_latency = network.pivot_table(
    index='checkpoint_type', columns='strategy', values='recovery_latency_sec'
)
pivot_latency.plot(kind='bar', ax=axes[0], width=0.6, edgecolor='white', colormap='muted')
axes[0].set_title('Recovery Latency by Storage Backend\n(Network Partition Scenario)', fontsize=12)
axes[0].set_ylabel('Recovery Latency (seconds)', fontsize=11)
axes[0].set_xlabel('Storage Backend', fontsize=11)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Strategy', fontsize=9)

pivot_dups = network.pivot_table(
    index='checkpoint_type', columns='strategy', values='duplicates_detected'
)
pivot_dups.plot(kind='bar', ax=axes[1], width=0.6, edgecolor='white', colormap='muted')
axes[1].set_title('Duplicate Records by Storage Backend\n(Network Partition Scenario)', fontsize=12)
axes[1].set_ylabel('Duplicate Records Detected', fontsize=11)
axes[1].set_xlabel('Storage Backend', fontsize=11)
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Strategy', fontsize=9)

plt.tight_layout()
plt.savefig('../experiments/fig4_storage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Finding: S3 backend produces significantly more duplicates and higher recovery latency vs local/HDFS.')

## 6. Silent Duplicates — Checkpoint Corruption Scenario

In [ ]:
corruption = df[
    (df['scenario'] == 'checkpoint_corruption') &
    (df['checkpoint_type'] == 'local')
].copy()

fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(
    corruption['strategy_label'],
    corruption['duplicates_detected'],
    color=colors, width=0.5, edgecolor='white'
)

for bar in bars:
    height = bar.get_height()
    label = f'{int(height)} duplicates' if height > 0 else 'None detected'
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.5,
        label, ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_title('Silent Duplicate Records Under Checkpoint Corruption', fontsize=14, pad=15)
ax.set_ylabel('Duplicate Records Detected', fontsize=12)
ax.set_xlabel('Checkpoint Strategy', fontsize=12)
ax.set_ylim(0, corruption['duplicates_detected'].max() * 1.3 + 5)
plt.tight_layout()
plt.savefig('../experiments/fig5_silent_duplicates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Finding: Strategy B produces silent duplicates under checkpoint corruption without Spark detection.')
print('Strategy C (WAL) recovers cleanly with zero duplicates.')

## 7. Key Findings Summary

In [ ]:
print('=' * 65)
print('KEY FINDINGS — Fault Tolerance Benchmark Summary')
print('=' * 65)

a_baseline = df[(df['strategy']=='A') & (df['scenario']=='baseline') & (df['checkpoint_type']=='local')]['normal_throughput_rps'].mean()
b_baseline = df[(df['strategy']=='B') & (df['scenario']=='baseline') & (df['checkpoint_type']=='local')]['normal_throughput_rps'].mean()
overhead = ((b_baseline - a_baseline) / b_baseline) * 100
print(f'\n1. Throughput cost of high-frequency checkpointing:')
print(f'   Strategy A: {a_baseline:,.0f} rec/s vs Strategy B: {b_baseline:,.0f} rec/s')
print(f'   Overhead: {overhead:.1f}% throughput reduction')

c_recovery = df[(df['strategy']=='C') & (df['scenario']=='node_failure') & (df['checkpoint_type']=='local')]['recovery_latency_sec'].mean()
a_recovery = df[(df['strategy']=='A') & (df['scenario']=='node_failure') & (df['checkpoint_type']=='local')]['recovery_latency_sec'].mean()
print(f'\n2. Recovery latency under node failure (local storage):')
print(f'   Strategy A: {a_recovery:.2f}s  |  Strategy C: {c_recovery:.2f}s')
print(f'   Strategy C is {((a_recovery - c_recovery)/a_recovery)*100:.1f}% faster to recover')

b_dups = df[(df['strategy']=='B') & (df['scenario']=='checkpoint_corruption') & (df['checkpoint_type']=='local')]['duplicates_detected'].sum()
c_dups = df[(df['strategy']=='C') & (df['scenario']=='checkpoint_corruption') & (df['checkpoint_type']=='local')]['duplicates_detected'].sum()
print(f'\n3. Silent duplicates under checkpoint corruption:')
print(f'   Strategy B: {int(b_dups)} duplicates (undetected by Spark)')
print(f'   Strategy C: {int(c_dups)} duplicates (WAL enables clean recovery)')

s3_latency = df[(df['scenario']=='network_partition') & (df['checkpoint_type']=='s3')]['recovery_latency_sec'].mean()
local_latency = df[(df['scenario']=='network_partition') & (df['checkpoint_type']=='local')]['recovery_latency_sec'].mean()
print(f'\n4. Storage backend impact under network partition:')
print(f'   S3 avg recovery: {s3_latency:.2f}s  |  Local avg recovery: {local_latency:.2f}s')
print(f'   S3 eventual consistency increases recovery latency by {((s3_latency-local_latency)/local_latency)*100:.0f}%')

print('\n' + '=' * 65)